# 前向传播实战

我们将利用已经学到的知识去完成三层神经网络的实现,们采用的数据集是 MNIST 手写数字图片集，输入节点数为 784，第一层的输出节点数是256，第二层的输出节点数是 128，第三层的输出节点是 10，也就是当前样本属于 10 类别的概率.

首先创建每个非线性层的𝑾和𝒃张量参数

In [2]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# 每层的张量都需要被优化，故使用 Variable 类型，并使用截断的正态分布初始化权值张量
# 偏置向量初始化为 0 即可
# 第一层的参数

w1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
b1 = tf.Variable(tf.zeros(256))

# 第二层参数
w2 = tf.Variable(tf.random.truncated_normal([256, 128], stddev=0.1))
b2 = tf.Variable(tf.zeros([128]))

# 第三层参数
w3 = tf.Variable(tf.random.truncated_normal([128, 10], stddev=0.1))
b3 = tf.Variable(tf.zeros([10]))

在前向计算时，首先将 shape 为[𝑏, 28,28]的输入张量的视图调整为[𝑏, 784]，即将每个
图片的矩阵数据调整为向量特征，这样才适合于网络的输入格式:

In [11]:
layers = tf.keras.layers
optimizers = tf.keras.optimizers
datasets = tf.keras.datasets

(x, y), _ = datasets.mnist.load_data()

x = 2 * tf.convert_to_tensor(x, dtype=tf.float32) / 255. - 1
y = tf.convert_to_tensor(y, tf.int32)
y = tf.one_hot(y, depth=10)

x = tf.reshape(x, [-1, 28*28])
print('y_shape:', y.shape)
x.shape

y_shape: (60000, 10)


TensorShape([60000, 784])

In [4]:
# 完成第一层的计算， [b, 784] @ [784, 256] + [256] -> [b, 256]
h1 = x @ w1 + tf.broadcast_to(b1, [x.shape[0], 256])
h1 = tf.nn.relu(h1)  # 经过激活函数
h1.shape

TensorShape([60000, 256])

In [5]:
# 完成第二、三层的计算
h2 = h1 @ w2 + tf.broadcast_to(b2, [h1.shape[0], 128])
h2 = tf.nn.relu(h2)

out = h2 @ w3 + tf.broadcast_to(b3, [h2.shape[0], 10])  # out层可以不需要激活函数

out.shape  

TensorShape([60000, 10])

In [10]:
# 计算网络输出与标签之间的均方差，mse = mean(sum(y-out)^2)
# [b, 10]
loss = tf.square(y - out)
loss = tf.reduce_mean(loss)
print(loss)

tf.Tensor(2.8058121, shape=(), dtype=float32)
